<span id="state-feedback과-observer"></span>

# State Feedback and Observer

<span id="학습-목표"></span>

## Learning objectives

Estimate velocity to control a system that measures only position, and compare estimated velocity and noise sensitivity.

<span id="측정값으로-상태를-구분할-수-있는가"></span>

## Can measurements distinguish states?

<span id="시작-조건"></span>

### Starting conditions

With position \[m\] and velocity \[m/s\] as states, use $\dot x=Ax+Bu$, $y=Cx$. Use $A=\begin{bmatrix}0&1\\-2&-3\end{bmatrix}$, $B=\begin{bmatrix}0\\1\end{bmatrix}$, $C=\begin{bmatrix}1&0\end{bmatrix}$.

<span id="진행-방법"></span>

### Procedure

1.  Form the controllability matrix \[B,AB\] and observability matrix \[C;CA\], and find their ranks.
2.  Compare the case where B is a zero vector with the case $C=\begin{bmatrix}0&0\end{bmatrix}$.
3.  Explain the distinction between an ineffective input and an unavailable measurement.

<span id="남길-결과"></span>

### Results to retain

- A comparison table of both ranks and feasible designs for the three conditions.

In [ ]:
import numpy as np
A=np.array([[0.,1.],[-2.,-3.]])
B=np.array([[0.],[1.]])
C=np.array([[1.,0.]])
for label,b,c in [('base',B,C),('no input',0*B,C),('no sensor',B,0*C)]:
    ctrb=np.column_stack([b,A@b]);obsv=np.vstack([c,c@A])
    print(label,np.linalg.matrix_rank(ctrb),np.linalg.matrix_rank(obsv))
# TODO: State what can and cannot be designed under each condition.


<span id="결과-기록자가-점검"></span>

## Record results and self-check

**Condition:**

**Controllability rank:**

**Observability rank:**

**Design feasibility:**

<span id="확인할-점"></span>

### Points to check

- Are both ranks of the baseline model 2?
- Did you avoid applying pole placement directly to a rank-deficient model?

<span id="state-feedback과-observer를-연결하기"></span>

## Connecting state feedback and observer

<span id="시작-조건-1"></span>

### Starting conditions

$u=-K\hat x+8r$, $r=1$. Control poles: −2,−4; observer poles: −8,−10. Use $x(0)=\begin{bmatrix}1\\0\end{bmatrix}$, $\hat x(0)=\begin{bmatrix}0\\0\end{bmatrix}$.

<span id="진행-방법-1"></span>

### Procedure

1.  Calculate K and L, and compare the eigenvalues of $A-BK$ and $A-LC$.
2.  Integrate actual and estimated states simultaneously, and plot position, velocity, and estimation errors over 0–4 s.
3.  Distinguish the reduction in estimation error from the output approaching its reference.

<span id="남길-결과-1"></span>

### Results to retain

- Two plots of actual and estimated position and actual and estimated velocity, and the final estimation errors.

In [ ]:
from scipy.signal import place_poles,StateSpace,lsim
import matplotlib.pyplot as plt
K=place_poles(A,B,[-2.,-4.]).gain_matrix
L=place_poles(A.T,C.T,[-8.,-10.]).gain_matrix.T
a_aug=np.block([[A,-B@K],[L@C,A-B@K-L@C]])
b_aug=np.vstack([8*B,8*B])
t=np.linspace(0,4,801)
_,z,_=lsim(StateSpace(a_aug,b_aug,np.eye(4),np.zeros((4,1))),np.ones(len(t)),t,X0=[1,0,0,0])
fig,axes=plt.subplots(2,1,sharex=True)
for j,ax in enumerate(axes):
    ax.plot(t,z[:,j],label='state');ax.plot(t,z[:,j+2],'--',label='estimate');ax.legend();ax.grid()
axes[0].set_ylabel('Position [m]');axes[1].set_ylabel('Velocity [m/s]');axes[-1].set_xlabel('Time [s]')
print('final estimation error:',np.linalg.norm(z[-1,:2]-z[-1,2:]))
# TODO: Print and compare the control poles and observer poles.


<span id="결과-기록자가-점검-1"></span>

## Record results and self-check

**K:**

**L:**

**Control poles:**

**Observer Pole:**

**Final error:**

<span id="확인할-점-1"></span>

### Points to check

- Do the eigenvalues of $A-BK$ and $A-LC$, after substituting the calculated $K$ and $L$, match the specified control and observer poles, respectively?
- At $t=4$ s, are the position and velocity estimation errors below 1e-6 m and 1e-6 m/s, respectively?

<span id="observer를-빠르게-하면-항상-좋은가"></span>

## Is a faster observer always better?

<span id="시작-조건-2"></span>

### Starting conditions

Add n(t)=0.02*sin(40*t) to the measurement. Compare observer poles (−8,−10) and (−20,−25), keeping controller K fixed.

<span id="진행-방법-2"></span>

### Procedure

1.  Use both observer settings in the practice notebook's function, and compare the control input u on the same time axis.
2.  Calculate position-error RMS, velocity-error RMS, and control-input RMS separately after 1 s.
3.  Interpret initial estimation-error convergence separately from the response to measurement noise.

<span id="남길-결과-2"></span>

### Results to retain

- Control-input curves, a comparison table of the three RMS values, and justification for the selected setting.

In [ ]:
def noisy_observer(poles):
    L=place_poles(A.T,C.T,poles).gain_matrix.T
    aa=np.block([[A,-B@K],[L@C,A-B@K-L@C]])
    bb=np.block([[8*B,np.zeros((2,1))],[8*B,L]])
    inputs=np.column_stack([np.ones(len(t)),.02*np.sin(40*t)])
    _,z,_=lsim(StateSpace(aa,bb,np.eye(4),np.zeros((4,2))),inputs,t,X0=[1,0,0,0])
    u=8-(z[:,2:]@K.T).ravel();keep=t>=1
    return u,np.sqrt(np.mean((z[keep,:2]-z[keep,2:])**2,axis=0)),np.sqrt(np.mean(u[keep]**2))
for poles in [(-8,-10),(-20,-25)]:
    u,err,rms=noisy_observer(poles);plt.plot(t,u,label=str(poles));print(poles,err,rms)
plt.xlabel('Time [s]');plt.ylabel('u [N]');plt.legend();plt.grid()
# TODO: Explain which of the two settings you would choose, based on the metrics.


<span id="결과-기록자가-점검-2"></span>

## Record results and self-check

**Observer Pole:**

**Position error RMS \[m\]:**

**Velocity error RMS \[m/s\]:**

**Input RMS \[N\]:**

**Reason for your choice:**

<span id="확인할-점-2"></span>

### Points to check

- Are noise, initial conditions, and time steps the same for both settings?
- Did you avoid conflating error and input in a single metric?

<span id="공식-예제데이터"></span>

## Official examples and data

- [SciPy · Pole placement](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.place_poles.html)